In [1]:
import pandas as pd
import numpy as np


# Загрузка данных
df = pd.read_csv("data/clickhouse_demo_github_events_data.csv")
df.head()

C:\Users\santiperro\AppData\Local\Temp\ipykernel_1016\1012236525.py:6: DtypeWarning: Columns (8,9,12,14,16,20,26,29,30,31,32,37,44,46,47,50,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/github_events_data.csv")


,file_time,event_type,actor_login,repo_name,created_at,updated_at,action,comment_id,body,path,...,diff_hunk,original_position,commit_id,original_commit_id,push_size,push_distinct_size,member_login,release_tag_name,release_name,review_state
0,2023-04-04 04:00:00,CommitCommentEvent,vercel[bot],OliverSpeir/portfolio,2023-04-04 04:59:04,2023-04-04 04:59:04,none,107427595.0,Successfully deployed to the following URLs:\n...,NaN,...,NaN,0.0,a12a8717acc6cc9aee74d8ac9ccf640ab6d134c0,NaN,0.0,0.0,NaN,NaN,NaN,none
1,2023-04-04 06:00:00,CommitCommentEvent,vercel[bot],OliverSpeir/portfolio,2023-04-04 06:41:53,2023-04-04 06:41:53,none,107437214.0,Successfully deployed to the following URLs:\n...,NaN,...,NaN,0.0,a411023e71d64d5f25d0c2e9d5162fde41069381,NaN,0.0,0.0,NaN,NaN,NaN,none
2,2023-04-04 09:00:00,CommitCommentEvent,vercel[bot],OliverSpeir/portfolio,2023-04-04 09:12:00,2023-04-04 09:12:00,none,107455321.0,Successfully deployed to the following URLs:\n...,NaN,...,NaN,0.0,d694b709ffec7fdafe31ca986db8ecae9ba10494,NaN,0.0,0.0,NaN,NaN,NaN,none
3,2023-04-04 09:00:00,CommitCommentEvent,vercel[bot],OliverSpeir/portfolio,2023-04-04 09:28:19,2023-04-04 09:28:19,none,107457449.0,Successfully deployed to the following URLs:\n...,NaN,...,NaN,0.0,b9d72c03c82e541260a29a8c8ecce59d8538f847,NaN,0.0,0.0,NaN,NaN,NaN,none
4,2023-04-05 01:00:00,CommitCommentEvent,vercel[bot],OliverSpeir/portfolio,2023-04-05 01:18:56,2023-04-05 01:18:56,none,107577128.0,Successfully deployed to the following URLs:\n...,NaN,...,NaN,0.0,32f96a4f1d84ec21782ac17110d975327bfd59b0,NaN,0.0,0.0,NaN,NaN,NaN,none


In [2]:
# Размерность данных
df.shape

(311070, 54)

In [3]:
# Столбцы
df.columns

Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at',
       'updated_at', 'action', 'comment_id', 'body', 'path', 'position',
       'line', 'ref', 'ref_type', 'creator_user_login', 'number', 'title',
       'labels', 'state', 'locked', 'assignee', 'assignees', 'comments',
       'author_association', 'closed_at', 'merged_at', 'merge_commit_sha',
       'requested_reviewers', 'requested_teams', 'head_ref', 'head_sha',
       'base_ref', 'base_sha', 'merged', 'mergeable', 'rebaseable',
       'mergeable_state', 'merged_by', 'review_comments',
       'maintainer_can_modify', 'commits', 'additions', 'deletions',
       'changed_files', 'diff_hunk', 'original_position', 'commit_id',
       'original_commit_id', 'push_size', 'push_distinct_size', 'member_login',
       'release_tag_name', 'release_name', 'review_state'],
      dtype='object')

In [4]:
# Обработка пустых значений и дубликатов

df.drop_duplicates(inplace=True)

df.replace('none', np.nan, inplace=True)
df.replace('[]', np.nan, inplace=True)
df.replace('NONE', np.nan, inplace=True)
df.replace('1970-01-01 00:00:00', np.nan, inplace=True)

df[["comment_id", "push_size", "push_distinct_size"]] = df[["comment_id", "push_size", "push_distinct_size"]].replace(0, np.nan)

filter1 = ~df["event_type"].isin(["CommitCommentEvent", "PullRequestReviewCommentEvent"])
df.loc[filter1, "position"] = df.loc[filter1, "position"].replace(0, np.nan)
df.loc[filter1, "line"] = df.loc[filter1, "line"].replace(0, np.nan)

filter2 = ~df["event_type"].isin(['Event', 'IssueCommentEvent', 'IssuesEvent', 'PullRequestEvent', 'PullRequestReviewCommentEvent', 'PullRequestReviewEvent'])
df.loc[filter2, "number"] = df.loc[filter2, "number"].replace(0, np.nan)

filter3 = ~df["event_type"].isin(['IssueCommentEvent', 'IssuesEvent', 'PullRequestEvent'])
df.loc[filter3, "comments"] = df.loc[filter3, "comments"].replace(0, np.nan)

filter4 = ~df["event_type"].isin(["PullRequestEvent"])
df.loc[filter4, "merged"] = df.loc[filter4, "merged"].replace(0, np.nan)
df.loc[filter4, "mergeable"] = df.loc[filter4, "mergeable"].replace(0, np.nan)
df.loc[filter4, "mergeable_state"] = df.loc[filter4, "mergeable_state"].replace('unknown', np.nan)
df.loc[filter4, "review_comments"] = df.loc[filter4, "review_comments"].replace(0, np.nan)
df.loc[filter4, "maintainer_can_modify"] = df.loc[filter4, "maintainer_can_modify"].replace(0, np.nan)
df.loc[filter4, "rebaseable"] = df.loc[filter4, "rebaseable"].replace(0, np.nan)
df.loc[filter4, "changed_files"] = df.loc[filter4, "changed_files"].replace(0, np.nan)

filter5 = ~df["event_type"].isin(['Event', 'PullRequestEvent'])
df.loc[filter5, "commits"] = df.loc[filter5, "commits"].replace(0, np.nan)
df.loc[filter5, "additions"] = df.loc[filter5, "additions"].replace(0, np.nan)
df.loc[filter5, "deletions"] = df.loc[filter5, "deletions"].replace(0, np.nan)

filter6 = ~df["event_type"].isin(["PullRequestReviewCommentEvent"])
df.loc[filter6, "original_position"] = df.loc[filter6, "original_position"].replace(0, np.nan)

filter7 = ~df["event_type"].isin(['IssuesEvent', 'PullRequestReviewEvent'])
df.loc[filter7, "locked"] = df.loc[filter7, "locked"].replace(0, np.nan)

In [5]:
# Количество записей каждого события

df.groupby(by="event_type")["event_type"].count() 

event_type
CommitCommentEvent               11821
CreateEvent                      13327
DeleteEvent                      12593
DownloadEvent                    17311
Event                            32586
FollowEvent                      35339
ForkApplyEvent                    3216
ForkEvent                        12404
GistEvent                         7489
GollumEvent                      15193
IssueCommentEvent                11134
IssuesEvent                      13041
MemberEvent                      13387
PublicEvent                      12355
PullRequestEvent                  9617
PullRequestReviewCommentEvent     8835
PullRequestReviewEvent            9695
PushEvent                        10809
ReleaseEvent                     16534
TeamAddEvent                     17180
WatchEvent                       18302
Name: event_type, dtype: int64

In [6]:
# Уберем устаревшие события
# Самая поздняя дата PublicEvent из целого датасета - 2017 год
# Event - 2011
# GistEvent - 2014
# DownloadEvent	- 2013
# FollowEvent - 2017
# ForkApplyEvent - 2012
# TeamAddEvent - 2014
# Нету SponsorshipEvent вообще

events = ['CommitCommentEvent', 'CreateEvent', 'DeleteEvent',
       'ForkEvent', 'GollumEvent', 'IssueCommentEvent',
       'IssuesEvent', 'MemberEvent', 'PullRequestEvent',
       'PullRequestReviewCommentEvent', 'PullRequestReviewEvent',
       'PushEvent', 'ReleaseEvent', 'WatchEvent']

In [7]:
# Ненулевые столбцы по каждому событию

not_null_columns_by_events = {}

for event in events:
    not_null_columns_by_events[event] = df.columns[df[df["event_type"] == event].any()]

not_null_columns_by_events

{'CommitCommentEvent': Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at',
        'updated_at', 'comment_id', 'body', 'path', 'position', 'line',
        'creator_user_login', 'commit_id'],
       dtype='object'),
 'CreateEvent': Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at',
        'ref', 'ref_type'],
       dtype='object'),
 'DeleteEvent': Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at',
        'ref', 'ref_type'],
       dtype='object'),
 'ForkEvent': Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at'], dtype='object'),
 'GollumEvent': Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at',
        'action'],
       dtype='object'),
 'IssueCommentEvent': Index(['file_time', 'event_type', 'actor_login', 'repo_name', 'created_at',
        'updated_at', 'action', 'comment_id', 'body', 'creator_user_login',
        'number', 'title', 'labels', 'state', 'assignee'

In [8]:
# Отделим данные каждого события друг от друга и отбросим ненужные столбцы

data_by_events = []

for event in not_null_columns_by_events.keys():
    data_by_events.append(df[df["event_type"] == event].dropna(axis=1, how="all"))

In [69]:
# Замыкание для удобного просмотра

def create_counter():
    count = 0
    def increment():
        nonlocal count
        if count < len(data_by_events) - 1:
            count += 1
        else:
            count = 0
        return count
    
    return increment

counter = create_counter()

In [114]:
# Используя замыкание, можно перезапускать ячейку для просмотра следующего события

data_by_events[counter()]

,file_time,event_type,actor_login,repo_name,created_at
131669,2016-11-22 03:00:00,ForkEvent,liangbaochang,CameloeAnthony/AndroidSdkSourceAnalysis,2016-11-22 03:33:31
131670,2017-07-05 07:00:00,ForkEvent,s347719,CameloeAnthony/AndroidSdkSourceAnalysis,2017-07-05 07:18:36
131671,2016-10-12 17:00:00,ForkEvent,supeijin,CameloeAnthony/Ant,2016-10-12 17:22:37
131672,2016-10-13 05:00:00,ForkEvent,NFLeo,CameloeAnthony/Ant,2016-10-13 05:23:47
131673,2016-10-13 07:00:00,ForkEvent,yuan1129,CameloeAnthony/Ant,2016-10-13 07:20:20
...,...,...,...,...,...
144068,2012-01-08 11:00:00,ForkEvent,hnit1998,concordiadesignlab/journaleditorialsystem,2012-01-08 11:12:09
144069,2022-03-02 22:00:00,ForkEvent,thiagodp,concordialang/ui-core,2022-03-02 22:46:04
144070,2019-07-28 02:00:00,ForkEvent,Sahana15Shankar,concordiaproj/APP_BattleShip_Project,2019-07-28 02:32:47
144071,2020-04-15 15:00:00,ForkEvent,arunipatel06,concordiaproj/APP_BattleShip_Project,2020-04-15 15:51:51


In [120]:
df[df["event_type"] == "PullRequestReviewEvent"].iloc[1000]

file_time                                              2023-05-05 08:00:00
event_type                                          PullRequestReviewEvent
actor_login                                                         wegank
repo_name                                                    NixOS/nixpkgs
created_at                                             2023-05-05 08:07:37
updated_at                                             2023-05-05 08:07:37
action                                                             created
comment_id                                                             NaN
body                     ###### Description of changes\r\n\r\nhttps://g...
path                                                                   NaN
position                                                               NaN
line                                                                   NaN
ref                                                                    NaN
ref_type                 